# Random Forest Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
sp = pd.read_csv('/content/drive/MyDrive/4733 project/data1/study_periods.csv')
print(sp.head(5))
print(sp.shape)
print(sp.columns.tolist())

   period train_start   train_end  test_start    test_end
0       1  1992-12-01  1995-11-16  1995-11-17  1996-11-12
1       2  1993-11-26  1996-11-12  1996-11-13  1997-11-07
2       3  1994-11-22  1997-11-07  1997-11-10  1998-11-05
3       4  1995-11-17  1998-11-05  1998-11-06  1999-11-03
4       5  1996-11-13  1999-11-03  1999-11-04  2000-10-30
(29, 5)
['period', 'train_start', 'train_end', 'test_start', 'test_end']


In [ ]:
"""
IEOR 4733 - ML Statistical Arbitrage on S&P 500
Random Forest Model Training (with checkpoint saving)

- Saves each period individually to Google Drive
- If interrupted, re-running will skip already-completed periods
- Final output merges all periods into rf_predictions.csv
"""

# ── Mount Google Drive ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import os
import time

# ── File Paths ─────────────────────────────────────────────────────────────────
DATA_DIR   = '/content/drive/MyDrive/4733 project/data1'
OUTPUT_DIR = '/content/drive/MyDrive/4733 project/data1'

# ── Feature columns (31 lagged returns) ───────────────────────────────────────
DAILY_LAGS   = list(range(1, 21))
MONTHLY_LAGS = list(range(40, 241, 20))
ALL_LAGS     = DAILY_LAGS + MONTHLY_LAGS
FEATURE_COLS = [f"R{lag}" for lag in ALL_LAGS]

# ── Random Forest Parameters ───────────────────────────────────────────────────
RF_PARAMS = {
    "n_estimators" : 1000,
    "max_depth"    : 20,
    "max_features" : "sqrt",
    "random_state" : 1,
    "n_jobs"       : -1,
    "class_weight" : "balanced",
}

# ══════════════════════════════════════════════════════════════════════════════
# Load Data
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("Loading data...")
print("=" * 60)

features = pd.read_csv(
    f"{DATA_DIR}/features.csv",
    parse_dates=["date"]
)

study_periods = pd.read_csv(
    f"{DATA_DIR}/study_periods.csv",
    parse_dates=["train_start", "train_end", "test_start", "test_end"]
)

print(f"  features.csv     : {len(features):,} rows")
print(f"  study_periods.csv: {len(study_periods)} periods")
print(f"  Feature columns  : {len(FEATURE_COLS)} ({FEATURE_COLS[0]} to {FEATURE_COLS[-1]})\n")

# ══════════════════════════════════════════════════════════════════════════════
# Train Random Forest — One Period at a Time with Checkpointing
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("Training Random Forest across all study periods...")
print("(Already completed periods will be skipped automatically)")
print("=" * 60)

all_predictions = []

for _, row in study_periods.iterrows():

    period      = int(row["period"])
    train_start = row["train_start"]
    train_end   = row["train_end"]
    test_start  = row["test_start"]
    test_end    = row["test_end"]

    # ── Checkpoint: skip if already done ──────────────────────────────────────
    checkpoint_path = f"{OUTPUT_DIR}/rf_pred_period_{period:02d}.csv"

    if os.path.exists(checkpoint_path):
        print(f"\nPeriod {period:02d}/{len(study_periods)}  |  "
              f"Already completed — loading from file...")
        saved = pd.read_csv(checkpoint_path, parse_dates=["date"])
        all_predictions.append(saved)
        print(f"  Loaded {len(saved):,} rows from {checkpoint_path}")
        continue

    # ── New period: train and predict ─────────────────────────────────────────
    print(f"\nPeriod {period:02d}/{len(study_periods)}"
          f"  |  Train: {train_start.date()} to {train_end.date()}"
          f"  |  Test : {test_start.date()} to {test_end.date()}")

    # Split
    train_mask = (features["date"] >= train_start) & (features["date"] <= train_end)
    test_mask  = (features["date"] >= test_start)  & (features["date"] <= test_end)

    train_data = features[train_mask].dropna(subset=FEATURE_COLS + ["Y"]).copy()
    test_data  = features[test_mask].dropna(subset=FEATURE_COLS).copy()

    X_train = train_data[FEATURE_COLS].values
    y_train = train_data["Y"].values
    X_test  = test_data[FEATURE_COLS].values

    print(f"  Train rows : {len(X_train):,}  "
          f"|  Test rows: {len(X_test):,}  "
          f"|  Positive rate: {y_train.mean():.3f}")

    # Train
    t0 = time.time()
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_train, y_train)
    elapsed = time.time() - t0
    print(f"  Training time: {elapsed:.1f}s  ({elapsed/60:.1f} min)")

    # Training Gini (used later for ENS2 ensemble weighting)
    train_proba = rf.predict_proba(X_train)[:, 1]
    train_auc   = roc_auc_score(y_train, train_proba)
    train_gini  = 2 * (train_auc - 0.5)
    print(f"  Train AUC : {train_auc:.4f}  |  Train Gini: {train_gini:.4f}")

    # Test predictions
    test_proba = rf.predict_proba(X_test)[:, 1]

    period_preds = test_data[["permno", "date", "Y"]].copy()
    period_preds["prob_RAF"]   = test_proba
    period_preds["period"]     = period
    period_preds["train_gini"] = train_gini

    # Directional accuracy
    dir_acc = ((test_proba >= 0.5) == period_preds["Y"].values).mean()
    print(f"  Test directional accuracy: {dir_acc:.4f}  (needs > 0.50)")

    # ── Save checkpoint immediately ────────────────────────────────────────────
    period_preds.to_csv(checkpoint_path, index=False)
    print(f"  Saved -> {checkpoint_path}")

    all_predictions.append(period_preds)

# ══════════════════════════════════════════════════════════════════════════════
# Merge All Periods and Save Final Output
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Merging all periods into final output...")
print("=" * 60)

rf_predictions = pd.concat(all_predictions, ignore_index=True)
rf_predictions = rf_predictions.sort_values(["date", "permno"]).reset_index(drop=True)

print(f"  Total rows : {len(rf_predictions):,}")
print(f"  Date range : {rf_predictions['date'].min().date()} "
      f"to {rf_predictions['date'].max().date()}")
print(f"  Periods    : {rf_predictions['period'].nunique()}")

# Preview
print("\n  Sample (first 5 rows):")
print(rf_predictions[["permno", "date", "prob_RAF", "Y", "period"]].head(5).to_string(index=False))

# Save merged file
final_path = f"{OUTPUT_DIR}/rf_predictions.csv"
rf_predictions.to_csv(final_path, index=False)
print(f"\n  Saved -> {final_path}")

# ══════════════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SUMMARY ACROSS ALL PERIODS")
print("=" * 60)

summary_rows = []
for _, row in study_periods.iterrows():
    period = int(row["period"])
    p_data = rf_predictions[rf_predictions["period"] == period]
    if len(p_data) == 0:
        continue
    dir_acc = ((p_data["prob_RAF"] >= 0.5) == p_data["Y"]).mean()
    summary_rows.append({
        "period"       : period,
        "test_start"   : row["test_start"].date(),
        "test_end"     : row["test_end"].date(),
        "test_rows"    : len(p_data),
        "train_gini"   : p_data["train_gini"].iloc[0].round(4),
        "dir_accuracy" : round(dir_acc, 4),
    })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
print(f"\nAverage directional accuracy : {summary['dir_accuracy'].mean():.4f}")
print(f"Average train Gini           : {summary['train_gini'].mean():.4f}")

print("\n" + "=" * 60)
print("RANDOM FOREST TRAINING COMPLETE")
print("=" * 60)
print(f"Individual period files : rf_pred_period_01.csv ... rf_pred_period_29.csv")
print(f"Final merged file       : rf_predictions.csv")
print(f"\nColumns in rf_predictions.csv:")
print(f"  permno     : stock identifier")
print(f"  date       : trading date")
print(f"  prob_RAF   : predicted probability of outperforming the market")
print(f"  Y          : actual label (1 = outperformed, 0 = did not)")
print(f"  period     : study period number")
print(f"  train_gini : training Gini index (used for ENS2 ensemble weighting)")
print(f"\nNext step: Backtest using prob_RAF to generate long/short signals")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...
  features.csv     : 4,115,161 rows
  study_periods.csv: 29 periods
  Feature columns  : 31 (R1 to R240)

Training Random Forest across all study periods...
(Already completed periods will be skipped automatically)

Period 01/29  |  Already completed — loading from file...
  Loaded 118,876 rows from /content/drive/MyDrive/4733 project/data1/rf_pred_period_01.csv

Period 02/29  |  Already completed — loading from file...
  Loaded 118,928 rows from /content/drive/MyDrive/4733 project/data1/rf_pred_period_02.csv

Period 03/29  |  Already completed — loading from file...
  Loaded 117,467 rows from /content/drive/MyDrive/4733 project/data1/rf_pred_period_03.csv

Period 04/29  |  Already completed — loading from file...
  Loaded 115,511 rows from /content/drive/MyDrive/4733 project/data1/rf_pred_period_04.csv

Period 05/29  |  Already completed — lo

In [ ]:
import pandas as pd
import os

DATA_DIR = '/content/drive/MyDrive/4733 project/data1'

# 读取所有已完成的period文件
all_preds = []
for i in range(1, 30):
    path = f"{DATA_DIR}/rf_pred_period_{i:02d}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=["date"])
        all_preds.append(df)
        print(f"  Loaded period {i:02d}  |  rows: {len(df):,}")
    else:
        print(f"  Period {i:02d} not found, skipping...")

# 合并
rf_predictions = pd.concat(all_preds, ignore_index=True)
rf_predictions = rf_predictions.sort_values(["date", "permno"]).reset_index(drop=True)

print(f"\n合并完成:")
print(f"  总行数  : {len(rf_predictions):,}")
print(f"  Period数: {rf_predictions['period'].nunique()}")
print(f"  日期范围: {rf_predictions['date'].min().date()} 到 {rf_predictions['date'].max().date()}")

# 保存
output_path = f"{DATA_DIR}/rf_predictions.csv"
rf_predictions.to_csv(output_path, index=False)
print(f"\n已保存 -> {output_path}")

  Loaded period 01  |  rows: 118,876
  Loaded period 02  |  rows: 118,928
  Loaded period 03  |  rows: 117,467
  Loaded period 04  |  rows: 115,511
  Loaded period 05  |  rows: 114,255
  Loaded period 06  |  rows: 113,170
  Loaded period 07  |  rows: 117,803
  Loaded period 08  |  rows: 120,618
  Loaded period 09  |  rows: 122,488
  Loaded period 10  |  rows: 121,030
  Loaded period 11  |  rows: 118,980
  Loaded period 12  |  rows: 117,280
  Loaded period 13  |  rows: 117,461
  Loaded period 14  |  rows: 116,281
  Loaded period 15  |  rows: 119,923
  Loaded period 16  |  rows: 122,169
  Loaded period 17  |  rows: 121,198
  Loaded period 18  |  rows: 120,736
  Loaded period 19  |  rows: 121,115
  Loaded period 20  |  rows: 121,017
  Loaded period 21  |  rows: 119,108
  Loaded period 22  |  rows: 119,889
  Loaded period 23  |  rows: 120,680
  Loaded period 24  |  rows: 121,000
  Loaded period 25  |  rows: 121,805
  Loaded period 26  |  rows: 122,117
  Loaded period 27  |  rows: 122,247
 